[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/legalintermedia/Physics/blob/main/Simulaciones_Jupyter/Laboratorio_Especializado/Fluidos_Especializado_4.ipynb)



# Laboratorio Especializado: Fluidos Especializado 4

Simulación computacional ultra-avanzada correspondiente a la literatura científica extendida (Nivel Experto).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def rayleigh_benard(nx=50, ny=25, prandtl=1.0, rayleigh=1e4, steps=1000, dt=1e-4):
    dx, dy = 1.0 / nx, 1.0 / ny
    T = np.zeros((ny, nx))
    T_old = np.zeros((ny, nx))
    psi = np.zeros((ny, nx))
    omega = np.zeros((ny, nx))
    omega_old = np.zeros((ny, nx))
    
    # Initialize Temperature with small perturbation
    T = 1.0 - np.linspace(0, 1, ny)[:, None] * np.ones((1, nx))
    T += 0.05 * np.random.randn(ny, nx)
    
    for step in range(steps):
        T_old[:] = T
        omega_old[:] = omega
        
        # Temperature advection-diffusion
        T[1:-1, 1:-1] = T_old[1:-1, 1:-1] + dt * (
            (T_old[1:-1, 2:] - 2*T_old[1:-1, 1:-1] + T_old[1:-1, :-2]) / dx**2 +
            (T_old[2:, 1:-1] - 2*T_old[1:-1, 1:-1] + T_old[:-2, 1:-1]) / dy**2
        )
        
        # Vorticity transport
        omega[1:-1, 1:-1] = omega_old[1:-1, 1:-1] + dt * (
            prandtl * ((omega_old[1:-1, 2:] - 2*omega_old[1:-1, 1:-1] + omega_old[1:-1, :-2]) / dx**2 +
                       (omega_old[2:, 1:-1] - 2*omega_old[1:-1, 1:-1] + omega_old[:-2, 1:-1]) / dy**2) +
            prandtl * rayleigh * (T[1:-1, 2:] - T[1:-1, :-2]) / (2 * dx)
        )
        
        # Solve Poisson for stream function (Jacobi iteration)
        for _ in range(20):
            psi[1:-1, 1:-1] = 0.25 * (psi[1:-1, 2:] + psi[1:-1, :-2] + psi[2:, 1:-1] + psi[:-2, 1:-1] + omega[1:-1, 1:-1] * dx * dy)
            
        # Update boundaries (simplified free-slip)
        T[0, :] = 1.0
        T[-1, :] = 0.0
        T[:, 0] = T[:, 1]
        T[:, -1] = T[:, -2]
        
    plt.figure(figsize=(10, 5))
    plt.contourf(T, levels=20, cmap='inferno')
    plt.colorbar(label='Temperature')
    plt.title('Rayleigh-Bénard Convection (Simplified)')
    plt.savefig('rayleigh_benard.png')

if __name__ == "__main__":
    rayleigh_benard()
